# Python-Spickzettel

Die wichtigsten Python-Befehle für die Auswertung im Praktikum, geordnet nach den Schritten einer Auswertung. Ausführlich erklärt werden sie im Notebook der Einführung [PYT](PYT/PYT.ipynb).

- Die Beispiele sind zum Kopieren gedacht. Ersetzen Sie die Variablennamen durch Ihre eigenen.
- Notation wie in den Versuchen: $\bar x$ Mittelwert, $s_x$ Standardabweichung, $u_x$ Messunsicherheit.
- **Tipp:** Ziehen Sie diesen Tab an den rechten Rand des Arbeitsbereichs. Dann sehen Sie Spickzettel und Ihr Notebook nebeneinander. Ein Inhaltsverzeichnis finden Sie in der linken Seitenleiste (Listen-Symbol).

### Pakete importieren

Die Blanko-Notebooks importieren in der ersten Code-Zelle bereits alles, was Sie für den jeweiligen Versuch brauchen.

```python
import numpy as np                      # Rechnen mit Arrays
import matplotlib.pyplot as plt         # Diagramme
from scipy.optimize import curve_fit    # Fit mit y-Unsicherheiten
from odrpack import odr_fit             # Fit mit x- und y-Unsicherheiten
import uncertainties.unumpy as unp      # automatische Fehlerfortpflanzung
```

## 1. Messwerte eingeben und einlesen

```python
# Messreihe als Array, direkt in SI-Einheiten umgerechnet
l = np.array([12.1, 12.3, 11.9, 12.2]) * 1e-2   # Länge: cm -> m
u_l = 0.1e-2                                     # Ableseunsicherheit in m

# Einzelwert mit Unsicherheit
m = 0.250                                        # Masse in kg
u_m = 0.001                                      # Unsicherheit in kg
```

Messdaten aus einer CSV-Datei, die im selben Ordner wie das Notebook liegt:

```python
daten = np.genfromtxt("messung.csv", delimiter=",", names=True)   # erste Zeile = Spaltennamen
print(daten.dtype.names)   # zeigt alle Spaltennamen
t = daten["t"]             # eine Spalte als Array
```

> **Achtung:** Dezimalzahlen mit Punkt schreiben: `12.1`, nicht `12,1`. Rechnen Sie alle Größen gleich zu Beginn in SI-Einheiten um, z. B. mm → m mit `* 1e-3`.

## 2. Kennwerte berechnen

```python
N = len(l)                     # Anzahl der Messwerte
l_mean = np.mean(l)            # Mittelwert
s_l = np.std(l, ddof=1)        # Standardabweichung der Stichprobe
u_l_mean = s_l / np.sqrt(N)    # Unsicherheit des Mittelwerts
```

> **Achtung:** `ddof=1` nicht vergessen. Ohne wird durch $N$ statt durch $N-1$ geteilt.

Rechnungen mit Arrays wirken auf jedes Element einzeln: Sind `s` und `t` gleich lange Arrays, liefert `s / t` für jedes Wertepaar eine Geschwindigkeit.

| Befehl | Bedeutung |
|--------|-----------|
| `np.min(x)`, `np.max(x)` | kleinster / größter Wert |
| `np.argmax(x)`, `np.argmin(x)` | Position (Index) des größten / kleinsten Werts |
| `x[0]`, `x[-1]`, `x[1:]` | erstes Element, letztes Element, alle ab dem zweiten |
| `np.abs(x)` | Betrag |
| `np.sqrt(x)`, `x**2` | Wurzel, Quadrat |
| `np.log(x)`, `np.exp(x)` | natürlicher Logarithmus, e-Funktion |
| `np.sin(x)`, `np.cos(x)`, `np.arctan(x)` | Winkelfunktionen, **im Bogenmaß** |
| `np.radians(w)`, `np.degrees(w)` | Grad → Bogenmaß, Bogenmaß → Grad |
| `np.pi` | $\pi$ |
| `np.interp(x0, x_tab, y_tab)` | Wert bei `x0` aus einer Tabelle linear interpolieren |

## 3. Fehlerfortpflanzung und Vergleich

Für eine Größe $f(a, b)$ aus den Messgrößen $a$ und $b$ gilt nach Gauß:

$$
u_f = \sqrt{\left(\frac{\partial f}{\partial a}\,u_a\right)^2 + \left(\frac{\partial f}{\partial b}\,u_b\right)^2}
$$

Beispiel: kinetische Energie $E = \frac{1}{2} m v^2$

```python
v = 3.20                       # Geschwindigkeit in m/s
u_v = 0.05                     # Unsicherheit in m/s
E = 0.5 * m * v**2             # Energie in J

dE_dm = 0.5 * v**2             # Ableitung nach m
dE_dv = m * v                  # Ableitung nach v
u_E = np.sqrt((dE_dm * u_m)**2 + (dE_dv * u_v)**2)
```

Vergleich mit einem Literatur- oder Sollwert:

```python
E_lit = 1.30                   # Vergleichswert in J
u_E_lit = 0.02                 # seine Unsicherheit in J

abw_rel = (E - E_lit) / E_lit * 100                  # relative Abweichung in %
abw_u = (E - E_lit) / np.sqrt(u_E**2 + u_E_lit**2)   # Abweichung in Vielfachen der Unsicherheit
```

Hat der Vergleichswert keine Unsicherheit, teilen Sie nur durch `u_E`.

## 4. Fehlerfortpflanzung automatisch: `uncertainties`

Das Paket `uncertainties` rechnet die Fehlerfortpflanzung selbst. Damit können Sie Ihre eigene Rechnung kontrollieren. Wo eine Aufgabe die Herleitung verlangt, ersetzt es diese nicht.

```python
m_u = unp.uarray(m, u_m)        # Wert und Unsicherheit verbinden
v_u = unp.uarray(v, u_v)        # geht genauso mit Arrays: unp.uarray(l, u_l)

E_u = 0.5 * m_u * v_u**2        # rechnen wie gewohnt

print(unp.nominal_values(E_u))  # Bestwert
print(unp.std_devs(E_u))        # Unsicherheit, ungerundet
```

> **Achtung:**
> - Wurzel, Logarithmus usw. mit `unp.sqrt`, `unp.log`, `unp.exp`, `unp.sin` … aufrufen. `np.sqrt` funktioniert mit Unsicherheiten nicht.
> - `print(E_u)` rundet die Unsicherheit kaufmännisch. Unsicherheiten werden aber aufgerundet. Lassen Sie sich die Unsicherheit deshalb mit `unp.std_devs` ausgeben und runden Sie selbst.

## 5. Darstellen

`x` und `y` sind gleich lange Arrays. Ihre Unsicherheiten `u_x` und `u_y` sind Arrays derselben Länge oder einzelne Werte.

```python
plt.errorbar(x, y, xerr=u_x, yerr=u_y, marker="o", linestyle="", label="Messwerte")

plt.xlabel(r"$t$ in $\mathrm{s}$")
plt.ylabel(r"$T$ in $\mathrm{K}$")
plt.title("Temperatur gegen Zeit")
plt.legend()
plt.grid()
plt.show()
```

| Befehl | Bedeutung |
|--------|-----------|
| `linestyle=""` | nur Punkte, keine Verbindungslinie |
| `r"$t$ in $\mathrm{s}$"` | Formeln in Beschriftungen, das `r` vor dem Text nicht vergessen |
| `plt.plot(x, y)` | Linie ohne Fehlerbalken |
| `plt.xlim(0, 10)`, `plt.ylim(0, 5)` | Achsenbereich festlegen |
| `plt.yscale("log")` | logarithmische y-Achse |
| `plt.axhline(y0)` | waagerechte Linie bei `y0`, z. B. für einen Literaturwert |

Alle Befehle für ein Diagramm gehören in dieselbe Zelle, `plt.show()` steht am Ende.

## 6. Fitten

| Unsicherheiten | Funktion | Modellfunktion bekommt |
|----------------|----------|------------------------|
| nur in y | `curve_fit` | `(x, m, b)`: die Parameter einzeln |
| in x und y | `odr_fit` | `(x, beta)`: die Parameter als Array |

```python
# curve_fit: nur y-Unsicherheiten
def linear_function(x, m, b):
    return m * x + b

popt, pcov = curve_fit(linear_function, x, y, sigma=u_y, absolute_sigma=True)
m, b = popt
u_m, u_b = np.sqrt(np.diag(pcov))
```

```python
# odr_fit: x- und y-Unsicherheiten
def linear_odr(x, beta):
    return beta[0] * x + beta[1]

output = odr_fit(linear_odr, x, y, beta0=[1, 0],
                 weight_x=1/u_x**2, weight_y=1/u_y**2)
m, b = output.beta
u_m, u_b = np.sqrt(np.diag(output.cov_beta))
```

- `beta0` sind Startwerte. Sie müssen nur grob stimmen, aber in der richtigen Größenordnung liegen, z. B. die Steigung aus dem ersten und letzten Messpunkt.
- `output.success` ist `True`, wenn der Fit konvergiert ist.
- `output.res_var` liegt bei etwa 1, wenn die Punkte im Rahmen ihrer Fehlerbalken streuen.

Fitkurve einzeichnen:

```python
x_fit = np.linspace(np.min(x), np.max(x), 200)    # dichte x-Werte für eine glatte Kurve

plt.errorbar(x, y, xerr=u_x, yerr=u_y, marker="o", linestyle="", label="Messwerte")
plt.plot(x_fit, linear_odr(x_fit, output.beta), label="Fit")
# bei curve_fit stattdessen: linear_function(x_fit, m, b)
plt.legend()
plt.show()
```

## 7. Ergebnisse ausgeben

```python
print(f"E = ({E:.3f} ± {u_E:.3f}) J")
print(f"Abweichung: {abw_rel:+.1f} % ({abw_u:+.1f} u)")
print(f"Steigung: {m * 1e3:.2f} mK/s")    # Einheit bei der Ausgabe umrechnen
```

| Format | Bedeutung | Beispiel |
|--------|-----------|----------|
| `{x:.2f}` | 2 Nachkommastellen | 3.14159 → `3.14` |
| `{x:.2e}` | wissenschaftliche Schreibweise | 0.00314 → `3.14e-03` |
| `{x:+.1f}` | mit Vorzeichen | 3.14159 → `+3.1` |
| `{x * 1e3:.1f}` | erst umrechnen, dann runden | 0.00314 → `3.1` |

`np.round(x, 2)` rundet alle Werte eines Arrays auf 2 Nachkommastellen, z. B. für eine Tabelle.